# H3: Can KryptoNet-40 be adapted to Kryptonite-40-Adapt?

**Claim refuted.** The mock paper finds that KryptoNet-40 used zero-shot, or with a linear or MLP probe on its frozen
representation, stays at chance on Kryptonite-40-Adapt, and concludes that the model cannot be adapted.

**Hypothesis H3.** LoRA on the four hidden layers reaches 90% mean test accuracy, while the frozen-representation
methods (and the same architecture trained from scratch) stay near chance.

The experiment itself is `run_lora.py` in the repository root (LoRA ranks 1--32, full fine-tuning, zero-shot,
linear probe, MLP probe and from-scratch, on 5 seeded 60/20/20 splits; Adam, lr $10^{-3}$, batch 128, at most 100
epochs, early stopping on validation accuracy with patience 15). This notebook runs it unchanged, logs its CPU time
for the sustainability section, and draws the report's figure from the numbers it saves (`results_lora.json`).

In [1]:
import os
os.environ["MKL_CBWR"] = "COMPATIBLE"  # before torch is imported: same numbers on every CPU (see run_lora.py)

import json
import runpy

import numpy as np
import matplotlib.pyplot as plt

from include.config import ROOT
from include.tracking import track
from include.figures import use_paper_style, savefig
from include.plots_h3 import plot_lora_rank, lora_ranks

use_paper_style()
RESULTS_FILE = ROOT / "results_lora.json"
PREDICTIONS = ROOT / "hiddenlabels" / "y_predicted_40_adapt.npy"
RERUN = not RESULTS_FILE.exists()  # set to True to rerun the experiment (about 15 CPU-minutes; rerun with the versions in requirements.txt)

## 1. Run the experiment

`run_lora.py` uses paths relative to the repository root, so it runs from there. It also rewrites the submitted
predictions; we keep a copy first and check whether the rerun reproduces them exactly.

In [2]:
if RERUN:
    backup = PREDICTIONS.read_bytes() if PREDICTIONS.exists() else None
    cwd = os.getcwd()
    os.chdir(ROOT)
    try:
        with track("H3_lora"):
            runpy.run_path("run_lora.py", run_name="__main__")
    finally:
        os.chdir(cwd)
    if backup is not None:
        print("\nrerun reproduces the submitted predictions exactly:", PREDICTIONS.read_bytes() == backup)
else:
    print(f"using saved results in {RESULTS_FILE.name} (set RERUN = True to retrain)")

labelled: (4000, 40) label mean: 0.513 | hidden: (10000, 40)
full model params: 809473 (expect 809473)
zero-shot acc on all 4000: 0.495 (paper reports ~0.48)


checks passed: LoRA starts identical to pretrained; trainable params = 28992
seed 0 zero_shot      trainable       0  val 0.486  test 0.499


seed 0 linear_probe   trainable     513  val 0.551  test 0.488


seed 0 scratch        trainable  809473  val 0.517  test 0.512


seed 0 full_finetune  trainable  809473  val 0.868  test 0.861


seed 0 lora_r1        trainable    3624  val 0.921  test 0.906


... (43 outputs left out)

seed 4 lora_r16       trainable   57984  val 0.955  test 0.959


seed 4 lora_r32       trainable  115968  val 0.950  test 0.954


seed 4 mlp_probe      trainable  263169  val 0.541  test 0.471

=== test accuracy, mean +/- std over 5 seeds ===
zero_shot      trainable       0  0.497 +/- 0.011
linear_probe   trainable     513  0.495 +/- 0.008
mlp_probe      trainable  263169  0.506 +/- 0.019
scratch        trainable  809473  0.496 +/- 0.018
full_finetune  trainable  809473  0.852 +/- 0.025
lora_r1        trainable    3624  0.939 +/- 0.021
lora_r2        trainable    7248  0.920 +/- 0.025
lora_r4        trainable   14496  0.943 +/- 0.017
lora_r8        trainable   28992  0.948 +/- 0.012
lora_r16       trainable   57984  0.956 +/- 0.004
lora_r32       trainable  115968  0.953 +/- 0.004

using r = 16 (a priori); validation-argmax would be lora_r16


saved hiddenlabels\y_predicted_40_adapt.npy (10000,) fraction predicted 1: 0.505
Now run: python check_submission.py

CPU time: 1303.4 s (0.362 CPU-hours)
[H3_lora] 541s, ~1.36 Wh, ~0.17 gCO2e

rerun reproduces the submitted predictions exactly: False


## 2. Results table (Table in the report)

Mean $\pm$ standard deviation of test accuracy over the 5 seeds, and the number of trainable parameters.

In [3]:
res = json.loads(RESULTS_FILE.read_text())
names = {"zero_shot": "Zero-shot", "linear_probe": "Linear probe", "mlp_probe": "MLP probe",
         "scratch": "Scratch (same arch.)", "full_finetune": "Full fine-tuning",
         **{f"lora_r{r}": f"LoRA r={r}" for r in lora_ranks(res)}}
print(f"{'method':<22}{'trainable':>10}   test accuracy      val accuracy")
for key, name in names.items():
    t, v = np.array(res["test"][key]), np.array(res["val"][key])
    print(f"{name:<22}{res['params'][key]:>10,}   {t.mean():.3f} +- {t.std():.3f}    {v.mean():.3f}")

lora = [k for k in names if k.startswith("lora")]

# numbers quoted in the report's text and figure caption
for key in lora:
    t = np.array(res["test"][key])
    print(f"{names[key]:<10} worst seed {t.min():.3f}, best seed {t.max():.3f}")
ft = np.array(res["test"]["full_finetune"])
print(f"full fine-tuning range {ft.min():.3f}--{ft.max():.3f}")
frozen = [np.mean(res["test"][k]) for k in ("zero_shot", "linear_probe", "mlp_probe", "scratch")]
print(f"frozen / scratch means {min(frozen):.3f}--{max(frozen):.3f}")
print("\nLoRA mean test accuracy above 90% at every rank:", all(np.mean(res["test"][k]) > 0.90 for k in lora))
print("rank with the highest mean validation accuracy:", max(lora, key=lambda k: np.mean(res["val"][k])))

method                 trainable   test accuracy      val accuracy
Zero-shot                      0   0.497 +- 0.011    0.485
Linear probe                 513   0.495 +- 0.008    0.522
MLP probe                263,169   0.506 +- 0.019    0.526
Scratch (same arch.)     809,473   0.496 +- 0.018    0.527
Full fine-tuning         809,473   0.852 +- 0.025    0.853
LoRA r=1                   3,624   0.939 +- 0.021    0.946
LoRA r=2                   7,248   0.920 +- 0.025    0.927
LoRA r=4                  14,496   0.943 +- 0.017    0.951
LoRA r=8                  28,992   0.948 +- 0.012    0.949
LoRA r=16                 57,984   0.956 +- 0.004    0.956
LoRA r=32                115,968   0.953 +- 0.004    0.955
LoRA r=1   worst seed 0.906, best seed 0.960
LoRA r=2   worst seed 0.877, best seed 0.942
LoRA r=4   worst seed 0.911, best seed 0.957
LoRA r=8   worst seed 0.925, best seed 0.957
LoRA r=16  worst seed 0.949, best seed 0.959
LoRA r=32  worst seed 0.946, best seed 0.957
full fine-tuni

## 3. Figure

LoRA test accuracy against rank (mean $\pm$ sd over 5 seeds), with full fine-tuning and the mean of the frozen /
from-scratch baselines as reference lines. Drawn one column wide with 10 pt text, as the report requires.

In [4]:
fig = plot_lora_rank(res)
savefig(fig, "H3_lora_rank")
plt.show()

<Figure size 325x250 with 1 Axes>